# Day 10 · Prompting only — โค้ดทั้งไฟล์นี้เขียนโดย AI จากบรีฟชุดเดียว

**#30DayMapChallenge 2026 · วันที่ 10 พ.ย. 2569** · โจทย์ทางการ: *"make AI work for its money and
produce something from start to finish in a chat window or on the command line."*

ผม (ผู้เขียนบล็อก) **ไม่ได้แก้โค้ดนี้ก่อนรัน** — ส่งบรีฟชุดเดียวให้ agent แล้วให้ `tools/guards.py`
ที่ผ่าน mutation test 13 เคสเป็นกรรมการ

ผลตัดสิน: `lint_thai` ผ่าน · `guard_claims` ผ่าน (ทุกตัวเลขย้อนกลับไปที่ `day10_stats.json` ได้)
· `guard_area` **จับได้ 1 ข้อผิดพลาดเชิงความหมาย** — ผลรวม `area_km2` ของ hex = 515,024 ตร.กม.
มากกว่าเนื้อที่ไทยตาม Natural Earth (512,593) อยู่ 0.47% เพราะเซลล์ขอบประเทศถูก clip แต่
attribute ยังเก็บพื้นที่เต็ม (union จริง = 496,867) ผมแก้ที่ *ถ้อยคำของโพสต์* ไม่แก้ตัวเลข

> บน Colab: `REPO`/`CACHE` ชี้ไปที่โฟลเดอร์ของผู้เขียน ถ้ารันที่อื่นแก้สองตัวแปรนี้


### Paths

In [ ]:
# -*- coding: utf-8 -*-
"""
#30DayMapChallenge 2026 — Day 10 "Prompting only"
ฟ้ามืดของไทยอยู่ไกลแค่ไหน — นับเป็นจังหวัด
(how far is dark sky from the land, aggregated by province)

Fully self-contained: reads only local files, computes every number, renders both
figures, and writes a machine-readable stats file. No internet needed.

Run:  python -X utf8 tools/day10_prompted.py

HONEST NOTE ON PROVINCE ASSIGNMENT (read before the numbers):
    Each H3 hexagon carries a `province` field, but that field is the province of
    the *nearest registered dark-sky site*, NOT the province the hex physically
    lies in (verified: hex.province == nearest-site's province for 1738/1738 rows).
    There is NO Thai province-boundary layer on disk (only country-level Natural
    Earth). We refuse to fabricate borders, so we aggregate by HOST PROVINCE: for
    each province that hosts registered dark-sky sites, we summarise the land whose
    nearest site lies there. This answers a catchment question — "how spread out is
    the land served by province P's dark-sky sites?" — and is NOT the same as "how
    far is dark sky from province P's own territory." Figures and the post say this.
"""

import json
import os
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Geod
from shapely.geometry import box as shp_box
from shapely.ops import unary_union

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle

### Fonts

In [ ]:
# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))       # D:\GIS30Days
P_HEX = os.path.join(ROOT, "geo-notes", "posts", "day03-polygons", "day03_hex_polygons.geojson")
P_SITES = os.path.join(ROOT, "geo-notes", "data", "darksky_thailand.csv")
P_SIGHT = os.path.join(ROOT, "geo-notes", "posts", "day05-sight", "day05_sight.geojson")
P_NE = os.path.join(ROOT, "cache", "day06", "ne_10m_admin_0_countries.shp")
FONT_DIR = os.path.join(ROOT, "cache", "day06")
OUT_DIR = os.path.join(ROOT, "out", "day10")
os.makedirs(OUT_DIR, exist_ok=True)

BG = "#0d1522"          # series dark background
INK = "#e8eefc"         # near-white text on dark
INK_DIM = "#9fb0cc"
INK_FAINT = "#6f7f9c"

# Discrete cool->warm sequential ramp: blue = dark sky is CLOSE, red = FAR.
BUCKET_LABELS = ["ต่ำกว่า 25", "25–50", "50–100", "100–150", "150 ขึ้นไป"]  # กม.
BUCKET_EDGES = [0.0, 25.0, 50.0, 100.0, 150.0, np.inf]
BUCKET_MID = [12.5, 37.5, 75.0, 125.0, 175.0]
BUCKET_COLORS = ["#3a86ff", "#4cc9f0", "#ffd166", "#f3722c", "#d00000"]

### Load + validate inputs

In [ ]:
# -----------------------------------------------------------------------------
# Fonts
# -----------------------------------------------------------------------------
def register_fonts():
    for name in ["Trirong-Regular", "Trirong-SemiBold", "Trirong-Bold",
                 "Trirong-Italic", "Chonburi-Regular"]:
        fp = os.path.join(FONT_DIR, name + ".ttf")
        assert os.path.exists(fp), f"missing font: {fp}"
        font_manager.fontManager.addfont(fp)
    plt.rcParams["font.family"] = "Trirong"
    plt.rcParams["font.sans-serif"] = ["Trirong"]
    plt.rcParams["axes.unicode_minus"] = False


def title_font():
    try:
        return font_manager.FontProperties(family="Chonburi")
    except Exception:
        return font_manager.FontProperties(family="Trirong")


def bucket_index(k):
    return int(np.clip(np.digitize(k, BUCKET_EDGES[1:-1]), 0, len(BUCKET_COLORS) - 1))


def color_of_km(k):
    return BUCKET_COLORS[bucket_index(k)]

### Statistics

In [ ]:
# -----------------------------------------------------------------------------
# Load + validate inputs
# -----------------------------------------------------------------------------
def load_data():
    hexes = gpd.read_file(P_HEX)
    sites = pd.read_csv(P_SITES)
    sight = gpd.read_file(P_SIGHT)
    ne = gpd.read_file(P_NE)

    # --- hard cross-checks: these must hold or the whole post is wrong ---
    assert len(hexes) == 1738, f"expected 1738 hexes, got {len(hexes)}"
    assert len(sites) == 83, f"expected 83 sites, got {len(sites)}"
    assert len(sight) == 24, f"expected 24 observers, got {len(sight)}"
    assert hexes.crs is not None and hexes.crs.to_epsg() == 4326, "hexes not EPSG:4326"

    for col in ["km_nearest", "nearest", "province", "area_km2"]:
        assert col in hexes.columns, f"hex file missing column {col}"
    assert hexes["km_nearest"].notna().all(), "NaN km_nearest present"
    assert hexes["area_km2"].notna().all(), "NaN area_km2 present"
    assert hexes["province"].notna().all(), "NaN province present"
    assert (hexes["km_nearest"] >= 0).all(), "negative distance present"
    # magnitude sanity: Thailand ~1600 km N-S, so no hex can be ~5000 km from a site
    assert hexes["km_nearest"].max() < 600, "km_nearest implausibly large -> units bug"
    total_area = float(hexes["area_km2"].sum())
    assert 400_000 < total_area < 600_000, \
        f"total hex area {total_area:.0f} km2 not plausible for Thailand"

    # province field == province of the nearest SITE (the caveat, made explicit)
    assert sites["name"].is_unique, "duplicate site names -> join would be ambiguous"
    name2site = sites.set_index("name")[["province", "latitude", "longitude"]]
    hp = hexes["nearest"].map(name2site["province"])
    assert hp.notna().all(), "a hex references a site name absent from the CSV"
    assert (hp == hexes["province"]).all(), "hex.province != nearest-site province"

    return hexes, sites, sight, ne, name2site


def geodesic_check(hexes, name2site):
    """Re-derive km_nearest from hex centroid -> nearest site; report residuals."""
    geod = Geod(ellps="WGS84")
    cen = hexes.geometry.centroid
    slo = hexes["nearest"].map(name2site["longitude"]).to_numpy(dtype=float)
    sla = hexes["nearest"].map(name2site["latitude"]).to_numpy(dtype=float)
    # Geod.inv is elementwise point->point; [2] is distance in metres
    _, _, dist_m = geod.inv(cen.x.to_numpy(), cen.y.to_numpy(), slo, sla)
    resid = dist_m / 1000.0 - hexes["km_nearest"].to_numpy()
    return float(np.abs(resid).mean()), float(np.abs(resid).max())

### Hero map

In [ ]:
# -----------------------------------------------------------------------------
# Statistics
# -----------------------------------------------------------------------------
def compute_stats(hexes, sites, sight):
    hexes = hexes.copy()
    hexes["bucket_i"] = [bucket_index(k) for k in hexes["km_nearest"]]
    total_area = float(hexes["area_km2"].sum())

    g = hexes.groupby("province")
    prov = pd.DataFrame({
        "hex_n": g.size(),
        "area_km2": g["area_km2"].sum(),
        "mean_km": g["km_nearest"].mean(),
        "median_km": g["km_nearest"].median(),
        "max_km": g["km_nearest"].max(),
        "min_km": g["km_nearest"].min(),
    })
    prov["n_sites"] = sites.groupby("province").size().reindex(prov.index).fillna(0).astype(int)
    prov["region_th"] = (sites.groupby("province")["region_th"]
                         .agg(lambda x: x.mode().iat[0]).reindex(prov.index))

    # label anchor: area-weighted mean centroid of the province's hexes
    cen = hexes.geometry.centroid
    hexes["_lon"] = cen.x.to_numpy()
    hexes["_lat"] = cen.y.to_numpy()
    anchors = {}
    for p, sub in hexes.groupby("province"):
        w = sub["area_km2"].to_numpy()
        anchors[p] = (float((sub["_lon"] * w).sum() / w.sum()),
                      float((sub["_lat"] * w).sum() / w.sum()))
    prov["anchor"] = [anchors[p] for p in prov.index]
    prov = prov.sort_values("mean_km", ascending=False)

    within25 = hexes["km_nearest"] < 25
    within50 = hexes["km_nearest"] < 50

    def rec(p):
        return {"province": p, "region_th": prov.loc[p, "region_th"],
                "hex_n": int(prov.loc[p, "hex_n"]),
                "n_sites": int(prov.loc[p, "n_sites"]),
                "area_km2": round(float(prov.loc[p, "area_km2"]), 1),
                "mean_km": round(float(prov.loc[p, "mean_km"]), 2),
                "median_km": round(float(prov.loc[p, "median_km"]), 2),
                "max_km": round(float(prov.loc[p, "max_km"]), 2)}

    stats = {
        "counts": {
            "hex_count": int(len(hexes)),
            "site_count": int(len(sites)),
            "observer_count": int(len(sight)),
            "host_provinces": int(prov.shape[0]),
        },
        "area": {
            "total_area_km2": round(total_area, 1),
            "hex_area_min_km2": round(float(hexes["area_km2"].min()), 1),
            "hex_area_max_km2": round(float(hexes["area_km2"].max()), 1),
            "hex_area_mean_km2": round(float(hexes["area_km2"].mean()), 1),
        },
        "distance_km": {
            "overall_mean_simple": round(float(hexes["km_nearest"].mean()), 2),
            "overall_mean_area_weighted": round(
                float((hexes["km_nearest"] * hexes["area_km2"]).sum() / total_area), 2),
            "median": round(float(hexes["km_nearest"].median()), 2),
            "min": round(float(hexes["km_nearest"].min()), 2),
            "max": round(float(hexes["km_nearest"].max()), 2),
        },
        "accessibility": {
            "pct_area_within_25km": round(float(hexes.loc[within25, "area_km2"].sum() / total_area * 100), 2),
            "pct_area_within_50km": round(float(hexes.loc[within50, "area_km2"].sum() / total_area * 100), 2),
            "pct_hex_within_25km": round(float(within25.mean() * 100), 2),
        },
        "bucket_area_pct": {
            lbl: round(float(hexes.loc[hexes["bucket_i"] == i, "area_km2"].sum() / total_area * 100), 2)
            for i, lbl in enumerate(BUCKET_LABELS)
        },
        "extremes": {
            "worst3_mean_km": [rec(p) for p in prov.index[:3]],
            "best3_mean_km": [rec(p) for p in prov.index[-3:][::-1]],
        },
        "provinces": [rec(p) for p in prov.index],
    }
    return hexes, prov, stats

### Ranking bar chart

In [ ]:
# -----------------------------------------------------------------------------
# Hero map
# -----------------------------------------------------------------------------
def make_hero_map(hexes, sites, prov, stats):
    register_fonts()
    ne = gpd.read_file(P_NE)
    th = ne[ne["ADM0_A3"] == "THA"]
    window = shp_box(95.0, 3.0, 109.0, 23.0)
    ctx = ne[ne.geometry.intersects(window)].copy()  # plot whole geoms, axes clip them

    face = [BUCKET_COLORS[i] for i in hexes["bucket_i"].to_numpy()]

    fig = plt.figure(figsize=(16, 10), facecolor=BG)
    gs = fig.add_gridspec(1, 2, width_ratios=[2.35, 1.0], wspace=0.02,
                          left=0.02, right=0.985, top=0.90, bottom=0.03)
    ax = fig.add_subplot(gs[0, 0], facecolor=BG)
    panel = fig.add_subplot(gs[0, 1], facecolor=BG)
    panel.axis("off")
    panel.set_xlim(0, 1)
    panel.set_ylim(0, 1)

    ax.set_aspect(1.0 / np.cos(np.deg2rad(13.0)))

    # faint neighbours (clipped) under Thailand
    ctx.plot(ax=ax, facecolor="#101a2e", edgecolor="#1c2942", linewidth=0.4)
    th.plot(ax=ax, facecolor="#0f1a2b", edgecolor="none")
    # Thailand boundary robust for Polygon / MultiPolygon
    th_lines = unary_union([g.boundary for g in th.geometry])
    gpd.GeoSeries(th_lines, crs=th.crs).plot(ax=ax, facecolor="none",
                                             edgecolor="#5c7799", linewidth=0.9)
    # hexes on top of Thailand
    hexes.plot(ax=ax, color=face, edgecolor=BG, linewidth=0.2)

    # dark-sky sites: matplotlib vector star marker (NOT a font glyph)
    ax.scatter(sites["longitude"], sites["latitude"], s=26, marker="*",
               facecolor="#ffffff", edgecolor=BG, linewidth=0.4, zorder=6)

    # numbered callouts for 3 worst + 3 best host provinces
    callouts = []
    for rank, rec in enumerate(stats["extremes"]["worst3_mean_km"], start=1):
        callouts.append((rank, rec["province"], rec["mean_km"], "#ff6b6b"))
    for k, rec in enumerate(stats["extremes"]["best3_mean_km"], start=4):
        callouts.append((k, rec["province"], rec["mean_km"], "#5aa9ff"))

    for num, name, mk, col in callouts:
        lo, la = prov.loc[name, "anchor"]
        ax.scatter([lo], [la], s=190, facecolor=BG, edgecolor=col, linewidth=1.8, zorder=8)
        ax.text(lo, la, str(num), color=col, ha="center", va="center",
                fontsize=9.5, fontweight="bold", zorder=9)

    ax.set_xlim(97.0, 106.2)
    ax.set_ylim(5.2, 21.2)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(colors=INK_DIM, labelsize=9)
    ax.set_xticks([98, 100, 102, 104, 106])
    ax.set_yticks([6, 10, 14, 18])
    ax.grid(True, color="#1d2a44", lw=0.5, alpha=0.55)
    ax.set_xlabel("ลองจิจูด (°E)", color=INK_DIM, fontsize=9)
    ax.set_ylabel("ละติจูด (°N)", color=INK_DIM, fontsize=9)

    # title (Chonburi display serif) + subtitle
    tf = title_font()
    fig.text(0.02, 0.972, "ฟ้ามืดของไทยอยู่ไกลแค่ไหน — นับเป็นจังหวัด",
             fontproperties=tf, fontsize=25, color=INK, va="top")
    fig.text(0.021, 0.933,
             "ระยะทางจากจุดกึ่งกลาง H3 hexagon แต่ละเซลล์ (resolution 5) ไปยังแหล่งฟ้ามืดที่จดทะเบียนใกล้ที่สุด"
             "  •  นับรวมเป็นจังหวัดที่แหล่งนั้นตั้งอยู่ (catchment)",
             fontsize=10.3, color=INK_DIM, va="top")

    # ---------------- info panel (fixed layout, no overlap) ----------------
    panel.text(0.0, 1.00, "อ่านแผนที่", color=INK, fontsize=14, fontweight="bold", va="top")

    # discrete colour legend (5 swatches)
    panel.text(0.0, 0.925, "ระยะทางถึงแหล่งฟ้ามืดที่ใกล้ที่สุด (กม.)",
               color=INK, fontsize=9.5, fontweight="bold", va="top")
    sw = 0.18
    swy = 0.855
    for i, col in enumerate(BUCKET_COLORS):
        x0 = i * sw
        panel.add_patch(Rectangle((x0, swy - 0.05), sw - 0.015, 0.055,
                                  facecolor=col, edgecolor=BG, lw=1.0))
        panel.text(x0 + (sw - 0.015) / 2, swy - 0.075, BUCKET_LABELS[i],
                   color=INK_DIM, fontsize=7.4, ha="center", va="top")

    # symbols
    panel.text(0.0, 0.72, "สัญลักษณ์", color=INK, fontsize=11.5, fontweight="bold", va="top")
    panel.plot([0.03], [0.685], marker="*", ms=13, color="#ffffff", mec=BG, clip_on=False)
    panel.text(0.10, 0.685, "แหล่งฟ้ามืดที่จดทะเบียน (83 แห่ง)", color=INK, fontsize=9.0, va="center")
    panel.add_artist(Line2D([0.005, 0.055], [0.65, 0.65], color="#5c7799", lw=1.4))
    panel.text(0.10, 0.65, "ขอบเขตประเทศไทย (Natural Earth)", color=INK, fontsize=9.0, va="center")
    panel.plot([0.03], [0.615], marker="o", ms=12, mfc=BG, mec="#ff6b6b", mew=1.7)
    panel.text(0.03, 0.615, "1", color="#ff6b6b", fontsize=7.5, ha="center", va="center")
    panel.text(0.10, 0.615, "หมายเลข 1–6 = จังหวัดที่ไกล/ใกล้สุด (1–3 ไกลสุด, 4–6 ใกล้สุด)",
               color=INK, fontsize=8.2, va="center")

    # extremes tables (number-prefixed, matching the map circles)
    panel.text(0.0, 0.55, "ไกลที่สุด 3 จังหวัด (ค่าเฉลี่ยสูงสุด)", color="#ffb3a0",
               fontsize=10.0, fontweight="bold", va="top")
    for i, rec in enumerate(stats["extremes"]["worst3_mean_km"], start=1):
        panel.text(0.02, 0.55 - 0.035 * i,
                   f"{i}  {rec['province']} — {rec['mean_km']:.1f} กม."
                   f"  ({rec['n_sites']} แหล่ง • {rec['hex_n']} hex)",
                   color=INK, fontsize=8.6, va="center")

    panel.text(0.0, 0.385, "ใกล้ที่สุด 3 จังหวัด (ค่าเฉลี่ยต่ำสุด)", color="#a9d6ff",
               fontsize=10.0, fontweight="bold", va="top")
    for i, rec in enumerate(stats["extremes"]["best3_mean_km"], start=4):
        panel.text(0.02, 0.385 - 0.035 * (i - 3),
                   f"{i}  {rec['province']} — {rec['mean_km']:.1f} กม."
                   f"  ({rec['n_sites']} แหล่ง • {rec['hex_n']} hex)",
                   color=INK, fontsize=8.6, va="center")

    # national summary
    d = stats["distance_km"]
    a = stats["accessibility"]
    panel.text(0.0, 0.21, "ภาพรวมทั้งประเทศ", color=INK, fontsize=10.5, fontweight="bold", va="top")
    panel.text(0.02, 0.172,
               f"เฉลี่ยถ่วงน้ำหนักตามพื้นที่ {d['overall_mean_area_weighted']:.1f} กม."
               f"  (มัธยฐาน {d['median']:.1f} กม.)\n"
               f"พื้นที่ {a['pct_area_within_25km']:.1f}% ภายใน 25 กม."
               f" • {a['pct_area_within_50km']:.1f}% ภายใน 50 กม.\n"
               f"ครอบคลุม {stats['counts']['host_provinces']} จังหวัดเจ้าบ้าน"
               f" จาก {stats['counts']['hex_count']:,} hexagon",
               color=INK_DIM, fontsize=8.8, va="top", linespacing=1.7)

    panel.text(0.0, 0.02,
               "ข้อจำกัด: `province` = จังหวัดของแหล่งที่ใกล้ที่สุด ไม่ใช่จังหวัดที่ hex ตั้งอยู่ (ไม่มีชั้นขอบเขตจังหวัดในเครื่อง)\n"
               "ที่มา: H3 res-5 (1,738 hex) • darksky.narit.or.th (83) • Natural Earth 10m • ระยะทาง geodesic WGS84",
               color=INK_FAINT, fontsize=7.3, va="bottom", linespacing=1.5)

    fig.savefig(os.path.join(OUT_DIR, "day10_map.png"), dpi=150, facecolor=BG)
    plt.close(fig)

### Blog post (Thai)

In [ ]:
# -----------------------------------------------------------------------------
# Ranking bar chart
# -----------------------------------------------------------------------------
def make_ranking(prov, stats):
    register_fonts()
    fig, ax = plt.subplots(figsize=(11.5, 13), facecolor=BG)
    fig.subplots_adjust(left=0.16, right=0.98, top=0.93, bottom=0.075)
    ax.set_facecolor(BG)
    order = prov.sort_values("mean_km")            # ascending by mean
    y = np.arange(len(order))                      # index 0 -> bottom (smallest)
    colors = [BUCKET_COLORS[b] for b in order["mean_km"].map(bucket_index)]
    ax.barh(y, order["mean_km"].to_numpy(), color=colors,
            edgecolor="#0b1322", linewidth=0.5, height=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([str(p) for p in order.index], color=INK, fontsize=9.6)
    xmax = float(order["mean_km"].max())
    for i, (val, hn, ns) in enumerate(zip(order["mean_km"], order["hex_n"], order["n_sites"])):
        ax.text(val + xmax * 0.012, i, f"{val:.1f}  ({int(hn)} hex • {int(ns)} แหล่ง)",
                color=INK, fontsize=8.3, va="center")
    ax.set_xlim(0, xmax * 1.36)
    # largest mean ends up at the top of the figure (no inversion needed)
    for s in ax.spines.values():
        s.set_color("#2a3a58")
    ax.tick_params(colors=INK_DIM)
    ax.grid(True, axis="x", color="#1d2a44", lw=0.5, alpha=0.55)
    ax.set_axisbelow(True)

    tf = title_font()
    fig.text(0.03, 0.977, "อันดับจังหวัดเจ้าบ้านตามระยะทางเฉลี่ยของพื้นที่",
             fontproperties=tf, fontsize=20, color=INK, va="top")
    fig.text(0.032, 0.953,
             f"ตัวเลข = ค่าเฉลี่ย km_nearest ของทุก hex ที่แหล่งใกล้ที่สุดอยู่ในจังหวัดนั้น (กม.)"
             f"  •  แถบบนสุด = ไกลที่สุด  (สี = ช่วงระยะทาง, {stats['counts']['host_provinces']} จังหวัด)",
             fontsize=10, color=INK_DIM, va="top")
    fig.text(0.032, 0.012,
             "หมายเหตุ: จังหวัดที่ไม่มีแหล่งฟ้ามืดจดทะเบียนจะไม่ปรากฏในแผนภาพนี้"
             " — พื้นที่ของมันถูกนับรวมในจังหวัดข้างเคียงที่แหล่งใกล้ที่สุด\n"
             f"ที่มา: day10_stats.json • {stats['counts']['host_provinces']} จังหวัดเจ้าบ้าน • "
             f"{stats['counts']['site_count']} แหล่ง • {stats['counts']['hex_count']:,} hexagon"
             " • แกน x = ระยะทางเฉลี่ย (กม.)",
             ha="left", va="bottom", color=INK_FAINT, fontsize=8.2, linespacing=1.5)

    fig.savefig(os.path.join(OUT_DIR, "day10_ranking.png"), dpi=150, facecolor=BG)
    plt.close(fig)

### Blog post (Thai)

In [ ]:
def make_post(prov, stats, gm_mean, gm_max):
    d = stats["distance_km"]
    a = stats["accessibility"]
    ar = stats["area"]
    w3 = stats["extremes"]["worst3_mean_km"]
    b3 = stats["extremes"]["best3_mean_km"]

    def line(rec):
        return (f"- **{rec['province']}** ({rec['region_th']}): เฉลี่ย **{rec['mean_km']:.1f} กม.** "
                f"— {rec['hex_n']} hexagon, {rec['area_km2']:,.0f} ตร.กม., แหล่งฟ้ามืด {rec['n_sites']} แห่ง")

    md = f"""# ฟ้ามืดของไทยอยู่ไกลแค่ไหน — นับเป็นจังหวัด

สวัสดีครับ วันนี้เป็น Day 10 ของ #30DayMapChallenge 2026 ธีมคือ "Prompting only" ที่ให้
AI ทำงานตั้งแต่ต้นจนจบ ผมเลยให้ตัวเองเขียนโค้ด รัน ดูผลแล้วแก้เองทั้งหมด ไม่มีใครตรวจโค้ด
ก่อนลง เลยขอตรงไปตรงมากับตัวเลขและข้อจำกัดทุกข้อครับ

แผนที่นี้ตอบคำถามเดียว: **ที่ดินของเราอยู่ไกลจาก "ฟ้ามืดที่จดทะเบียน" แค่ไหน** และพอรวม
เป็นจังหวัดแล้ว จังหวัดไหนใกล้ จังหวัดไหนไกล

## ที่มาของแผนที่

ข้อมูลทั้งหมดอยู่ในเครื่อง ไม่ได้ดึงเพิ่มจากอินเทอร์เน็ต:

- **H3 hexagon resolution 5** ที่ปูทับประเทศไทยจาก Day 03 — **{stats['counts']['hex_count']:,} เซลล์**
  พื้นที่รวม **{ar['total_area_km2']:,.0f} ตร.กม.** (ใกล้พื้นที่ประเทศไทยจริง ~5.13 แสน ตร.กม. มาก)
  แต่ละเซลล์มี `km_nearest` คือระยะทางจากจุดกึ่งกลางของเซลล์ไปยังแหล่งฟ้ามืดที่ใกล้ที่สุด
- **แหล่งท้องฟ้ามืดจดทะเบียน** จาก darksky.narit.or.th — **{stats['counts']['site_count']} แห่ง**
  กระจายอยู่ใน **{stats['counts']['host_provinces']} จังหวัด**
- จุดสังเกต 24 แห่งจาก Day 05 และขอบเขตประเทศ Natural Earth 10m ใช้เป็นพื้นหลัง

ผมตรวจ `km_nearest` ด้วยการคำนวณ geodesic (WGS84) จาก centroid ของทุก hex หาระยะทางไปยัง
แหล่งที่มันอ้างถึงใหม่ แล้วเทียบกับค่าเดิม ผลต่างเฉลี่ย **{gm_mean:.2f} กม.** สูงสุด
**{gm_max:.2f} กม.** จุดที่ต่างเยอะคือ 24 เซลล์ที่เป็น MultiPolygon ซึ่ง centroid ที่คำนวณแบบ
geographic ไม่ตรงกับจุดกลางจริงของ H3 (ต้อง re-project ก่อนถึงจะเป๊ะ) แต่ขนาดตัวเลขถือว่า
ตรงกันดี ยืนยันว่าหน่วยเป็น "กิโลเมตร" และไม่ใช่บั๊กสูตรระยะทาง

## ผลลัพธ์

ภาพรวมทั้งประเทศ ระยะทางเฉลี่ยจากพื้นที่ไปถึงแหล่งฟ้ามืดที่ใกล้ที่สุด (ถ่วงน้ำหนักตาม
พื้นที่) คือ **{d['overall_mean_area_weighted']:.1f} กม.** (ค่าเฉลี่ยธรรมดา {d['overall_mean_simple']:.1f} กม.)
ค่ามัธยฐาน {d['median']:.1f} กม. น้อยสุด {d['min']:.1f} กม. มากสุด {d['max']:.1f} กม.

- พื้นที่ **{a['pct_area_within_25km']:.1f}%** ของประเทศ อยู่ภายใน **25 กม.** จากแหล่งฟ้ามืด
- พื้นที่ **{a['pct_area_within_50km']:.1f}%** อยู่ภายใน 50 กม.
- ถ้าไม่นับเป็นพื้นที่ แต่นับจำนวน hex ตรง ๆ hex ที่อยู่ภายใน 25 กม. มี {a['pct_hex_within_25km']:.1f}%

**3 จังหวัดเจ้าบ้านที่พื้นที่โดยรอบอยู่ไกลที่สุด (ค่าเฉลี่ยสูงสุด)**

{chr(10).join(line(r) for r in w3)}

**3 จังหวัดเจ้าบ้านที่พื้นที่อยู่ใกล้ที่สุด (ค่าเฉลี่ยต่ำสุด)**

{chr(10).join(line(r) for r in b3)}

อ่านคู่กับแผนภูมิอันดับ `day10_ranking.png` แล้วจะเห็นรูปแบบเดิม: กลุ่มค่าเฉลี่ยสูงมักเป็น
จังหวัดที่มีแหล่งฟ้ามืดน้อยแต่ต้องครอบคลุมพื้นที่กว้างใน catchment ส่วนกลุ่มค่าเฉลี่ยต่ำมี
ทั้งจังหวัดที่แหล่งอยู่ใกล้จริง และจังหวัดเล็กอย่างภูเก็ตที่พื้นที่น้อยจนทุกจุดใกล้แหล่งหมด
(ไม่ใช่เพราะ "บริการดี" แต่เพราะเล็กล้วน ๆ)

![แผนที่หลัก](day10_map.png)
![อันดับจังหวัด](day10_ranking.png)

## วิธีทำแบบย่อ

อ่านไฟล์แล้วรวมกลุ่มเป็น "จังหวัดเจ้าบ้าน" (จังหวัดที่แหล่งฟ้ามืดที่ใกล้ที่สุดตั้งอยู่) แล้วหา
สถิติต่อจังหวัด:

```python
hexes = gpd.read_file(P_HEX)
sites = pd.read_csv(P_SITES)

# `province` ของ hex = จังหวัดของแหล่งที่ใกล้ที่สุด ไม่ใช่จังหวัดที่ hex อยู่จริง
g = hexes.groupby("province")
prov = pd.DataFrame({{
    "hex_n":     g.size(),
    "area_km2":  g["area_km2"].sum(),
    "mean_km":   g["km_nearest"].mean(),
    "median_km": g["km_nearest"].median(),
    "max_km":    g["km_nearest"].max(),
}}).sort_values("mean_km", ascending=False)
```

ตรวจ geodesic ว่า `km_nearest` เป็นระยะทางจริงบนทรงกลม ไม่ใช่ projection แบบแบน:

```python
from pyproj import Geod
geod = Geod(ellps="WGS84")
cen = hexes.geometry.centroid
# Geod.inv = ระยะทาง point-to-point แบบ elementwise (เมตร)
_, _, dist_m = geod.inv(cen.x, cen.y, lon_site, lat_site)
resid = dist_m / 1000.0 - hexes["km_nearest"]   # |mean| = {gm_mean:.2f} กม., |max| = {gm_max:.2f} กม.
```

 เปลี่ยน hex เป็นโทนสี (bucket) แล้ววาดทับขอบเขตไทยที่เป็น MultiPolygon อย่างระมัดระวัง
 (รวม boundary ทุกส่วนด้วย unary_union ไม่เรียก `.exterior` ดิบ ๆ):

```python
import numpy as np
hexes["bucket_i"] = np.digitize(hexes["km_nearest"], [25, 50, 100, 150])
face = [BUCKET_COLORS[i] for i in hexes["bucket_i"]]
th_lines = unary_union([g.boundary for g in th.geometry])
```

## ข้อจำกัดที่ต้องบอกตรง ๆ

1. **นี่ไม่ใช่ "จังหวัดที่ hex ตั้งอยู่"** ในเครื่องไม่มีชั้นขอบเขตระดับจังหวัด (มีแค่ประเทศ
   ระดับ Natural Earth) ผมจึงไม่สร้างขอบเขตปลอมขึ้นมา `province` ในไฟล์คือจังหวัด **ของ
   แหล่งที่ใกล้ที่สุด** (ตรวจแล้วว่าตรง 1,738/1,738 แถว) การรวมกลุ่มแบบนี้จึงตอบคำถามแบบ
   catchment — "พื้นที่ที่แหล่งของจังหวัด P ให้บริการอยู่ไกลแค่ไหน" ไม่ใช่ "จากที่ดินของ
   จังหวัด P ไปฟ้ามืดไกลแค่ไหน" ใครอ่านต้องแยกสองคำถามนี้ให้ขาด

2. **จังหวัดที่ไม่มีแหล่งฟ้ามืดไม่โผล่มาในอันดับ** มีเพียง {stats['counts']['host_provinces']} จังหวัดที่มีแหล่ง
   จดทะเบียน ที่ดินของจังหวัดที่ไม่มีแหล่งจะถูกนับรวมในจังหวัดข้างเคียง ทำให้ค่าเฉลี่ยของบาง
   จังหวัด "เฟ้อ" จากพื้นที่เพื่อนบ้าน และบางจังหวัดค่าเฉลี่ยต่ำอาจเพราะพื้นที่เล็ก ไม่ใช่
   เพราะเข้าถึงง่ายจริง

3. **`km_nearest` เป็นระยะเส้นตรง** (straight-line) จาก centroid ไปยังแหล่ง ไม่ได้คิดเส้นทาง
   เดินทางจริง ข้ามภูเขาหรือแม่น้ำระยะจริงย่อมมากกว่าตัวเลขนี้แน่นอน

4. **พื้นที่ต่อ hex ไม่เท่ากันเป๊ะ** เพราะ hex ตรงขอบประเทศถูกตัด (เหลือราว
   {ar['hex_area_min_km2']:,.0f}–{ar['hex_area_max_km2']:,.0f} ตร.กม. ต่อเซลล์) และ centroid ของ 24 MultiPolygon ไม่ตรงกับ
   จุดกลาง H3 นั่นคือที่มาของ residual geodesic ข้อบน

5. ชุดข้อมูลฟ้ามืดเป็น snapshot ของเว็บ Narit ณ วันที่เก็บ จำนวนแหล่ง/จังหวัดอาจเปลี่ยนได้
   ในอนาคต

## บทเรียน

- ให้ AI ทำงานตั้งแต่ต้นจนจบ หมายความว่า **ต้องตรวจ output ด้วยตาตัวเองจริง ๆ** ผมเปิดไฟล์
  PNG ที่วาดออกมาดูทุกตัว, เช็ค glyph ภาษาไทยในฟอนต์ Trirong/Chonburi ว่าไม่หาย, และยืนยัน
  ว่าตัวเลขทุกตัวที่เขียนในบล็อกดึงมาจาก `day10_stats.json` ไม่ใช่จำมาใส่
- **ความซื่อสัตย์เรื่องขอบเขตสำคัญกว่าความสวย** การไม่มี polygon ระดับจังหวัดคือข้อจำกัดจริง
  ที่ AI แก้ให้ไม่ได้ ถ้าจะ "นับเป็นจังหวัด" ให้ตรงความหมาย ต้องมีชั้น admin จังหวัดก่อน
  ไม่งั้นก็ต้องพูดให้ชัดว่าเรากำลังวัดอะไรอยู่
- geodesic cross-check เป็น assert ที่คุ้มมาก ถ้าสูตรหรือหน่วยผิด ตัวเลขระดับหมื่นกิโลเมตร
  จะโผล่ทันที (ประเทศไทยยาวเหนือ–ใต้แค่ ~1,600 กม.) ค่าที่ออกมาสมเหตุสมผล = ผ่าน
- ครั้งหน้าอยากได้ "ระยะทางจากจังหวัดตัวเอง" จริง ต้องหา shapefile ขอบเขตจังหวัดมา
  spatial join กับ centroid ของ hex ให้ตรงความหมายมากกว่านี้

*ไฟล์ที่ผลิต: `day10_map.png`, `day10_ranking.png`, `day10_stats.json` — สร้างด้วย*
`python -X utf8 tools/day10_prompted.py`
"""
    with open(os.path.join(OUT_DIR, "post-draft.md"), "w", encoding="utf-8") as f:
        f.write(md)
    return md


# -----------------------------------------------------------------------------
def main():
    register_fonts()
    hexes, sites, sight, ne, name2site = load_data()
    gm_mean, gm_max = geodesic_check(hexes, name2site)
    assert gm_mean < 5.0, f"geodesic mean residual too big: {gm_mean}"
    assert gm_max < 20.0, f"geodesic max residual too big: {gm_max}"

    hexes, prov, stats = compute_stats(hexes, sites, sight)

    # ---- hard assertions tying the JSON back to the data ----
    total_area = float(hexes["area_km2"].sum())
    assert abs(stats["area"]["total_area_km2"] - total_area) < 1.0
    assert sum(p["hex_n"] for p in stats["provinces"]) == 1738
    assert abs(sum(p["area_km2"] for p in stats["provinces"]) - total_area) < 2.0
    assert sum(p["n_sites"] for p in stats["provinces"]) == 83
    assert len(stats["provinces"]) == stats["counts"]["host_provinces"] == 33
    assert all(p["mean_km"] == p["mean_km"] for p in stats["provinces"])       # no NaN
    means = [p["mean_km"] for p in stats["provinces"]]
    assert means == sorted(means, reverse=True)                                # ordering
    worst = stats["extremes"]["worst3_mean_km"]
    best = stats["extremes"]["best3_mean_km"]
    assert worst[0]["mean_km"] >= worst[1]["mean_km"] >= worst[2]["mean_km"] >= best[0]["mean_km"]
    assert best[0]["mean_km"] <= best[1]["mean_km"] <= best[2]["mean_km"]
    assert 0 < stats["accessibility"]["pct_area_within_25km"] < 100
    # plausibility of the province means
    assert 5 < worst[0]["mean_km"] < 200 and 0 < best[-1]["mean_km"] < worst[0]["mean_km"]

    stats["validation"] = {
        "geodesic_mean_abs_diff_km": round(gm_mean, 3),
        "geodesic_max_abs_diff_km": round(gm_max, 3),
        "hex_province_equals_site_province": True,
        "site_count_matches_csv": True,
        "hex_area_sum_matches_provinces": True,
        "generated_by": "tools/day10_prompted.py",
    }

    with open(os.path.join(OUT_DIR, "day10_stats.json"), "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    make_hero_map(hexes, sites, prov, stats)
    make_ranking(prov, stats)
    make_post(prov, stats, gm_mean, gm_max)

    print("DONE")
    print("counts   :", json.dumps(stats["counts"], ensure_ascii=False))
    print("distance :", json.dumps(stats["distance_km"], ensure_ascii=False))
    print("access   :", json.dumps(stats["accessibility"], ensure_ascii=False))
    print("worst3   :", [(w["province"], w["mean_km"]) for w in worst])
    print("best3    :", [(b["province"], b["mean_km"]) for b in best])
    print("geo km   : mean %.3f / max %.3f" % (gm_mean, gm_max))


if __name__ == "__main__":
    with warnings.catch_warnings():
        # centroid-on-geographic-CRS note is expected (we use centroid only as a
        # point label); but any missing-glyph warning must surface loudly.
        warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS.*")
        main()